# 14 · A human approval gate with interrupt and resume
Level L3 · Part 3 · core · ~25 min · builds on 06, 09

## Goal
Pause a refund before it is paid, wait for a person, and resume the same run safely. The refund is paid once, even
when the run restarts or the resume is called twice.

## Why it matters
Lesson 09 held the refund in a Python list, `PENDING`. That list lives inside one process. Run the block below: the
refund goes out as soon as it is asked for, with no pause and no person. A retry after a timeout pays it again.
**Nothing in this block waits for anyone.**

In [1]:
REFUNDS = []  # what the money has really done


def naive_refund(order_id: str, amount_usd: float) -> str:
    REFUNDS.append((order_id, amount_usd))  # the money moves here, with no pause
    return "refunded"


# The first reply times out after the money has moved. The agent cannot tell, so it asks again.
naive_refund("2207", 84.00)
naive_refund("2207", 84.00)  # the retry: nothing remembers the first call
print("refunds made:", len(REFUNDS), "· paid out:", sum(a for _, a in REFUNDS), "USD for an 84.00 USD order")

refunds made: 2 · paid out: 168.0 USD for an 84.00 USD order


**What it makes easy:** a refund waits as saved state, not as a running program. The reviewer can come back an hour
later, a restart loses nothing, and a repeated resume cannot pay twice. **When you don't need it:** a read-only tool
changes nothing, so it needs no gate. A write under a small limit may be allowed without a person; the rule is yours.

## The idea
An approval gate is a **saved pause**. The agent writes the request to a store with the status `waiting_approval`,
and the run ends. A person decides later, in their own process. Then a **resume** reads the store and does what the
status says. Two rules make it safe. The decision is a **conditional update**, so only one decision can win. The
payment has an **idempotency key**, the run id, so a second payment with the same key does nothing.

Analogy: a cheque with a second signature. The clerk puts it in a tray, not in the till. The manager signs it later.
The clerk checks the tray before paying, and a cheque stamped "paid" is never paid again.

## Picture
```mermaid
sequenceDiagram
  participant A as Agent run (process 1)
  participant S as Saved state (approvals.db)
  participant R as Reviewer (process 2)
  participant Z as Resume (process 3)
  A->>S: propose refund: status waiting_approval
  A-->>A: run ends, nothing waits in memory
  R->>S: decide: approved (only if still waiting)
  Z->>S: resume: status approved, pay once with the run id as key
  Z->>S: status paid
```

## Step by step
**Step 1 · the store and the rules (setup, no call).** The gate is a small file with three commands: `propose`,
`decide` and `resume`. Each runs in a **new Python process**, so every step is a real restart. All state is in a
SQLite file from the standard library. Orders are fictional, as in lessons 06 to 09. A refund above $50 needs a senior
reviewer: that is the escalation rule. The `idem_key` of `ledger` is a PRIMARY KEY, so a duplicate insert is silently
skipped. The cell deletes the file first, so every run starts clean.

In [2]:
from pathlib import Path

Path("../data/14").mkdir(parents=True, exist_ok=True)           # run-time files for this lesson
Path("../data/14/approvals.db").unlink(missing_ok=True)         # start with no saved runs

The next cell is a `%%writefile` cell: its first line is the Jupyter magic, and it saves the rest of the cell as
`data/14/refund_gate.py`. The cell runs no Python. Statuses move `waiting_approval` → `approved` → `paying` → `paid`,
or to `rejected`. `paying` lets a crashed run be finished safely. Run the cell before Step 2.

In [3]:
%%writefile ../data/14/refund_gate.py
"""Refund approval gate: save the request, wait for a person, pay once. State lives in approvals.db."""
import sqlite3
import sys
from contextlib import closing
from pathlib import Path

DB = Path(__file__).with_name("approvals.db")  # saved state, next to this file
ORDERS = {"1093": 45.00, "2207": 84.00}        # fictional order totals, as in lessons 06 to 09
SENIOR_LIMIT_USD = 50.00                       # above this amount, only senior-lead may decide


def sql(stmt: str, args=()):
    """One statement in its own transaction. Returns (rows, rows changed)."""
    with closing(sqlite3.connect(DB)) as db, db:  # the inner 'with' commits, or rolls back on error
        cur = db.execute(stmt, args)
        return cur.fetchall(), cur.rowcount


sql("CREATE TABLE IF NOT EXISTS runs (run_id TEXT PRIMARY KEY, order_id TEXT, amount_usd REAL, "
    "status TEXT, decided_by TEXT, result TEXT)")
sql("CREATE TABLE IF NOT EXISTS ledger (idem_key TEXT PRIMARY KEY, order_id TEXT, amount_usd REAL)")


def status_of(run_id: str):
    rows, _ = sql("SELECT status, order_id, amount_usd, decided_by, result FROM runs WHERE run_id = ?", (run_id,))
    return rows[0] if rows else None


def propose(run_id: str, order_id: str, amount: str) -> str:
    if order_id not in ORDERS or float(amount) > ORDERS[order_id]:
        return f"refused: order {order_id} is unknown or the amount is over its total"
    try:
        sql("INSERT INTO runs VALUES (?, ?, ?, 'waiting_approval', '', '')", (run_id, order_id, float(amount)))
    except sqlite3.IntegrityError:
        return f"refused: run {run_id} already exists"
    return f"{run_id}: waiting for approval, nothing paid"


def decide(run_id: str, reviewer: str, approve: bool) -> str:
    row = status_of(run_id)
    if row is None:
        return "no such run"
    if row[0] != "waiting_approval":
        return f"already decided: {row[0]}"
    if row[2] > SENIOR_LIMIT_USD and reviewer != "senior-lead":  # escalation
        return f"refused: {row[2]:.2f} USD needs senior-lead"
    new = "approved" if approve else "rejected"
    # conditional update: only a run still waiting can be decided, so one decision wins
    _, changed = sql("UPDATE runs SET status = ?, decided_by = ? WHERE run_id = ? AND status = 'waiting_approval'",
                     (new, reviewer, run_id))
    return f"{run_id}: {new} by {reviewer}" if changed == 1 else "already decided by someone else"


def pay(run_id: str) -> str:
    # claim: only an approved run moves to 'paying', and only one caller can claim it
    _, claimed = sql("UPDATE runs SET status = 'paying' WHERE run_id = ? AND status = 'approved'", (run_id,))
    status, order_id, amount, _, _ = status_of(run_id)
    if claimed != 1 and status != "paying":  # 'paying' without a claim is a crash after the claim: finish it
        raise RuntimeError(f"run {run_id} is not approved, so it cannot be paid")
    # the key is the run id: a repeat insert is ignored, so the money moves once
    sql("INSERT OR IGNORE INTO ledger VALUES (?, ?, ?)", (run_id, order_id, amount))
    result = f"paid {amount:.2f} USD for order {order_id}"
    sql("UPDATE runs SET status = 'paid', result = ? WHERE run_id = ?", (result, run_id))
    return result


def resume(run_id: str) -> str:
    row = status_of(run_id)  # everything the run needs is read from the store, not from memory
    if row is None:
        return "no such run"
    if row[0] == "waiting_approval":
        return "still waiting for a person; nothing paid"
    if row[0] == "rejected":
        return f"rejected by {row[3]}; nothing paid"
    if row[0] in ("approved", "paying"):
        return pay(run_id)
    return f"already paid, no second refund: {row[4]}"


if __name__ == "__main__":
    cmd, *args = sys.argv[1:]
    if cmd == "decide":
        args[2] = args[2] == "approve"  # "approve" or "reject" becomes True or False
    print({"propose": propose, "decide": decide, "resume": resume}[cmd](*args))

Overwriting ../data/14/refund_gate.py


**Step 2 · the helpers (setup, no call).** `gate` runs one command in a new process and returns what it printed.
`query` and `peek` read the saved rows, so you can see the state between two processes. The kernel imports the same
file, so the Try it and the mistakes below can use its names.

In [4]:
import sqlite3
import subprocess
import sys

GATE = Path("../data/14/refund_gate.py").resolve()
DB_PATH = GATE.with_name("approvals.db")
sys.path.insert(0, str(GATE.parent))
import refund_gate  # the same file, in this kernel: its constants and the error below


def gate(*args) -> str:
    """Run one gate command in a NEW Python process: a restart between steps."""
    out = subprocess.run([sys.executable, str(GATE), *args], capture_output=True, text=True)
    return (out.stdout + out.stderr).strip()


def query(stmt: str, args=()) -> list:
    with sqlite3.connect(DB_PATH) as db:  # commits any change, such as the simulated crash in Step 6
        return db.execute(stmt, args).fetchall()


def peek(run_id: str) -> list:
    return query("SELECT run_id, status, amount_usd, decided_by FROM runs WHERE run_id = ?", (run_id,))

**Step 3 · process 1: the agent proposes the refund.** A customer asks for a refund. The model is given one tool,
`propose_refund`, and no tool that pays. It makes one call (the course's default model, recorded once). The
proposal goes into the store and the run ends. Nothing is paid, and nothing waits in memory.

In [5]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("14_human-approval-gate")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's default model
client = genai.Client(vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
                      http_options=types.HttpOptions(timeout=30_000))

LLM mode: replay (14_human-approval-gate.yaml)


The one tool the model sees, and the instructions that say a person approves the refund.

In [6]:
PROPOSE = types.FunctionDeclaration(
    name="propose_refund", description="Propose a refund. A person must approve it before any money moves.",
    parameters_json_schema={"type": "object", "properties": {"order_id": {"type": "string"},
                            "amount_usd": {"type": "number"}}, "required": ["order_id", "amount_usd"]})
config = types.GenerateContentConfig(
    system_instruction="You are the support assistant for Brightside Outfitters. You cannot pay refunds: "
                       "call propose_refund, and a person approves it.",
    tools=[types.Tool(function_declarations=[PROPOSE])], temperature=0)

The customer's message goes to the model, which answers with a tool request. Its arguments are the proposal.

In [7]:
CUSTOMER = "My parcel for order 2207 arrived with a torn box. Please refund the 84.00 USD I paid."
reply = client.models.generate_content(model=MODEL, contents=CUSTOMER, config=config)
llm_replay.stop()  # the only model call of this notebook
call = next(p.function_call for p in reply.candidates[0].content.parts if p.function_call)
args = dict(call.args)
print("model asked:", call.name, args)
print(gate("propose", "r-1001", args["order_id"], str(args["amount_usd"])))
print("saved row:", peek("r-1001"))  # the store keeps the amount as a float, so 84 shows as 84.0
print("ledger (payments made):", query("SELECT * FROM ledger"))

model asked: propose_refund {'order_id': '2207', 'amount_usd': 84}
r-1001: waiting for approval, nothing paid
saved row: [('r-1001', 'waiting_approval', 84.0, '')]
ledger (payments made): []


**Fallback (only if the model cell was skipped).** The same proposal is written with no model, so Steps 4 to 6 still
run. It is not model output: the arguments are typed in by hand, as the model would have sent them.

In [8]:
if peek("r-1001") is None:
    print(gate("propose", "r-1001", "2207", "84.00"))
print("saved row:", peek("r-1001"))

saved row: [('r-1001', 'waiting_approval', 84.0, '')]


**Step 4 · process 2: a resume before anyone decides.** The run is resumed in a fresh process, as a worker would
after a restart. It reads the store, sees `waiting_approval`, and stops. Nothing is paid.

In [9]:
print(gate("resume", "r-1001"))
print("ledger (payments made):", query("SELECT * FROM ledger"))

still waiting for a person; nothing paid
ledger (payments made): []


**Step 5 · processes 3 and 4: escalation, approval and the payment.** The support desk's own reviewer tries first. The
amount is above $50, so the decision is refused. A senior lead approves. Then the run is resumed: this is the
payment, and it happens once.

In [10]:
print(gate("decide", "r-1001", "agent-desk-1", "approve"))
print(gate("decide", "r-1001", "senior-lead", "approve"))
print(gate("resume", "r-1001"))
print("saved row:", peek("r-1001"), "· ledger:", query("SELECT * FROM ledger"))

refused: 84.00 USD needs senior-lead
r-1001: approved by senior-lead


paid 84.00 USD for order 2207
saved row: [('r-1001', 'paid', 84.0, 'senior-lead')] · ledger: [('r-1001', '2207', 84.0)]


**Step 6 · a repeat, a reject, and a crash in the middle of the payment.** A resume of the paid run pays nothing
again. A reviewer rejects a third refund, and its resume says so. Then a fourth run is made to crash after the
payment is claimed but before it is recorded: the row is left in `paying`. A resume finishes it with the same key,
so that run pays exactly once.

In [11]:
print(gate("resume", "r-1001"))
print(gate("propose", "r-1003", "1093", "45.00"))
print(gate("decide", "r-1003", "agent-desk-1", "reject"))
print(gate("resume", "r-1003"))
print(gate("propose", "r-1002", "1093", "45.00"))
print(gate("decide", "r-1002", "agent-desk-1", "approve"))
query("UPDATE runs SET status = 'paying' WHERE run_id = 'r-1002'")  # simulate the crash: claimed, not recorded
print(gate("resume", "r-1002"))
print(gate("resume", "r-1002"))
print("ledger (one row per paid run):", query("SELECT * FROM ledger"))

already paid, no second refund: paid 84.00 USD for order 2207


r-1003: waiting for approval, nothing paid


r-1003: rejected by agent-desk-1
rejected by agent-desk-1; nothing paid


r-1002: waiting for approval, nothing paid


r-1002: approved by agent-desk-1


paid 45.00 USD for order 1093


already paid, no second refund: paid 45.00 USD for order 1093
ledger (one row per paid run): [('r-1001', '2207', 84.0), ('r-1002', '1093', 45.0)]


## Under the hood
The gate needs three things, and the store has them all. The **saved state** is the `runs` row, so a new process
can read it. The **conditional update** (`WHERE status = 'waiting_approval'`) lets only one decision win, because
the database checks and changes the row in one step. The **idempotency key** is the primary key of `ledger`, so a
second insert with the same key is ignored. The demo uses a scratch database, so the gate's rows stay as they are.

In [12]:
db = sqlite3.connect(":memory:")  # a scratch database for this demo only
db.execute("CREATE TABLE runs (run_id TEXT PRIMARY KEY, status TEXT)")
db.execute("CREATE TABLE ledger (idem_key TEXT PRIMARY KEY, amount_usd REAL)")
db.execute("INSERT INTO runs VALUES ('demo', 'waiting_approval')")
for reviewer in ("agent-desk-1", "second-reviewer"):  # only the first decision matches the WHERE clause
    cur = db.execute("UPDATE runs SET status = 'approved' WHERE run_id = 'demo' AND status = 'waiting_approval'")
    print(reviewer, "decides -> rows changed:", cur.rowcount)  # 1, then 0
cur = db.execute("UPDATE runs SET status = 'paying' WHERE run_id = 'demo' AND status = 'approved'")
print("claim the payment -> rows changed:", cur.rowcount)  # 1: only one caller can claim it
for _ in range(2):  # the same key twice: the primary key makes the second insert a no-op
    cur = db.execute("INSERT OR IGNORE INTO ledger VALUES ('demo', 84.0)")
    print("ledger insert -> rows changed:", cur.rowcount)  # 1, then 0
db.close()

agent-desk-1 decides -> rows changed: 1
second-reviewer decides -> rows changed: 0
claim the payment -> rows changed: 1
ledger insert -> rows changed: 1
ledger insert -> rows changed: 0


## At work
- Production refunds wait in an approval queue, and the reviewer sees the exact arguments that will run. The
  approver's identity is logged with the decision → lesson 31 (agent identity and the audit log).
- A restart must neither lose a pending approval nor pay twice. Teams keep the run state outside the worker and
  give each payment an idempotency key. → lesson 32 (idempotency and retries) and lesson 35 (durable runs).
- Frameworks give the same pause as a primitive: LangGraph's `interrupt()` saves the run with a checkpointer and
  resumes it later. Lesson 19 builds the support desk on it. Go deeper: Temporal, in lesson 35.

## Common mistakes
1. **Calling the payment before the approval.** `pay` checks the status in the same conditional update as the
   claim. Here a rejected run (r-1003, Step 6) is passed to it, and it refuses.

In [13]:
refund_gate.pay("r-1003")  # r-1003 was rejected in Step 6: pay refuses a run that is not approved

RuntimeError: run r-1003 is not approved, so it cannot be paid

The error names the run, which makes the failure easy to trace in a log.

2. **The agent approving its own write.** Keep `decide` out of the agent's tools. It runs under the reviewer's
   identity.
3. **Keeping the pause in memory.** A Python list is lost when the process ends (lesson 09's `PENDING`).
4. **Paying with no key.** Without the run id as the key, a retry inserts a second payment row.

## Try it
Write `needs_senior(amount_usd)` that returns `True` when the amount is above the gate's `SENIOR_LIMIT_USD`, and
`False` otherwise. Check it on 45.00, 50.00 and 84.00. Exactly 50.00 does not need a senior, because the rule is
"above $50". No model call is needed.

In [14]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [15]:
def needs_senior(amount_usd: float) -> bool:
    return amount_usd > refund_gate.SENIOR_LIMIT_USD


for amount in (45.00, 50.00, 84.00):
    print(f"{amount:6.2f} USD -> senior needed: {needs_senior(amount)}")

 45.00 USD -> senior needed: False
 50.00 USD -> senior needed: False
 84.00 USD -> senior needed: True


## Recap
- A refund request is saved as `waiting_approval` and the run ends. A restart loses nothing.
- A person decides in another process. A conditional update lets only one decision win, and an amount above the
  limit needs a senior.
- Resume reads the saved state. The payment is claimed once, under the run id as its idempotency key.
- The agent proposes and resumes; a person decides. Keep `decide` out of the agent's tools.

**Interview:** "A refund needs a manager's approval, and the server can restart at any time. How do you build it?"
Strong answer: "The request is stored as waiting for approval, and the run ends. The manager's decision is a
conditional update, and resume reads the stored status. The run id is the payment's idempotency key, so a restart or
a second resume cannot pay twice."